In [6]:
from pathlib import Path

DATA_DIR = Path("..") / "data"

for f in DATA_DIR.rglob("*"):
    if f.is_file():
        print(f.name, round(f.stat().st_size / 1024**3, 2), "GB")

digikala-comments.csv 1.19 GB
digikala-products.csv 0.24 GB


In [7]:
import duckdb

con = duckdb.connect()
COMMENTS = "../data/digikala-comments.csv"

In [8]:
con.sql(f"""
    DESCRIBE
    SELECT * FROM read_csv('{COMMENTS}', all_varchar=true)
""").show()

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ title                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ body                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ created_at            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rate                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ recommendation_status │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_buyer              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ advantages            │ VARCHAR     │ 

In [10]:
n_rows = con.sql(f"""
    SELECT COUNT(*) FROM read_csv('{COMMENTS}', all_varchar=true)
""").fetchall()[0][0]

print(f"{n_rows:,}")

6,156,289


In [11]:
con.sql(f"""
    SELECT
        COUNT(*)                    AS n_rows,
        COUNT(DISTINCT id)          AS unique_ids,
        COUNT(DISTINCT product_id)  AS unique_products,
        COUNT(DISTINCT seller_code) AS unique_sellers
    FROM read_csv('{COMMENTS}', all_varchar=true)
""").show()

┌─────────┬────────────┬─────────────────┬────────────────┐
│ n_rows  │ unique_ids │ unique_products │ unique_sellers │
│  int64  │   int64    │      int64      │     int64      │
├─────────┼────────────┼─────────────────┼────────────────┤
│ 6156289 │    6153060 │          331599 │          29214 │
└─────────┴────────────┴─────────────────┴────────────────┘



In [12]:
con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE id IS NULL)     AS null_ids,
        COUNT(*) FILTER (WHERE id IS NOT NULL) AS non_null_ids,
        COUNT(DISTINCT id)                     AS unique_ids
    FROM read_csv('{COMMENTS}', all_varchar=true)
""").show()

┌──────────┬──────────────┬────────────┐
│ null_ids │ non_null_ids │ unique_ids │
│  int64   │    int64     │   int64    │
├──────────┼──────────────┼────────────┤
│        0 │      6156289 │    6153060 │
└──────────┴──────────────┴────────────┘



In [13]:
con.sql(f"""
    WITH per_product AS (
        SELECT product_id, COUNT(*) AS n_reviews
        FROM read_csv('{COMMENTS}', all_varchar=true)
        WHERE product_id IS NOT NULL
        GROUP BY product_id
    )
    SELECT
        COUNT(*)                       AS n_products,
        MIN(n_reviews)                 AS min_reviews,
        quantile_cont(n_reviews, 0.5)  AS median_reviews,
        AVG(n_reviews)                 AS mean_reviews,
        quantile_cont(n_reviews, 0.99) AS p99_reviews,
        MAX(n_reviews)                 AS max_reviews
    FROM per_product
""").show()

┌────────────┬─────────────┬────────────────┬────────────────────┬─────────────┬─────────────┐
│ n_products │ min_reviews │ median_reviews │    mean_reviews    │ p99_reviews │ max_reviews │
│   int64    │    int64    │     double     │       double       │   double    │    int64    │
├────────────┼─────────────┼────────────────┼────────────────────┼─────────────┼─────────────┤
│     331599 │           1 │            4.0 │ 18.565463104532885 │       200.0 │         400 │
└────────────┴─────────────┴────────────────┴────────────────────┴─────────────┴─────────────┘



## A1 Findings

- The comments file has 6,156,289 records and 15 columns (1.19 GB), counted with a CSV parser.
- No record has a missing id, but there are only 6,153,060 distinct ids, so 3,229 records reuse an id seen earlier (to be investigated).
- The data covers 331,599 distinct products and 29,214 distinct seller codes.
- Reviews per product are highly skewed: min 1, median 4, mean 18.6, 99th percentile 200, max 400.
- Implication: the largest product holds under 0.01% of all records, so a product-grouped split can reach the 70/15/15 target closely.